# Projeto QSAR — Dataset: **BBBP**
**Disciplina:** Aprendizado Profundo | **Professor:** Rosalvo Neto
**Semente aleatória:** `42`

## Contexto
A barreira hematoencefálica (BHE) controla o que entra no cérebro. Prever permeabilidade é essencial para fármacos do SNC.

**Rótulos:** `class_0` = Não penetra a BHE | `class_1` = Penetra a BHE


## 0. Instalação (descomente se necessário)

In [ ]:
# !pip install tensorflow scikit-learn matplotlib seaborn pandas

## 1. Imports e Semente Aleatória

In [ ]:
import os, random, time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"TensorFlow {tf.__version__} | Seed = {SEED}")

## 2. Caminhos do Dataset
> **Ajuste `BASE_DIR`** para o diretório onde `dataset_BBBP` está localizado.


In [ ]:
# Ajuste conforme seu ambiente:
# Colab: BASE_DIR = '/content/drive/MyDrive/Projeto2/dataset_BBBP'
# Local: BASE_DIR = '/caminho/Projeto2/dataset_BBBP'
BASE_DIR  = './dataset_BBBP'

TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR   = os.path.join(BASE_DIR, 'val')
TEST_DIR  = os.path.join(BASE_DIR, 'test')

for name, d in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    ok = os.path.isdir(d)
    print(f"{name}: {d} — {'OK' if ok else 'NAO ENCONTRADO'} ")

## 3. Análise Exploratória (EDA)

In [ ]:
def count_imgs(split_dir):
    result = {}
    for cls in sorted(os.listdir(split_dir)):
        p = os.path.join(split_dir, cls)
        if os.path.isdir(p):
            result[cls] = len([f for f in os.listdir(p) if f.lower().endswith('.png')])
    return result

LABEL = {'class_0': 'Não penetra a BHE', 'class_1': 'Penetra a BHE'}

print(f"=== Dataset: BBBP ===")
for split, d in [('TRAIN', TRAIN_DIR), ('VAL', VAL_DIR), ('TEST', TEST_DIR)]:
    c = count_imgs(d)
    total = sum(c.values())
    print(f"\n[{split}] Total: {total}")
    for cls, n in c.items():
        pct = n/total*100
        print(f"  {cls} ({LABEL[cls]}): {n} ({pct:.1f}%)")

In [ ]:
# Amostras visuais
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('Amostras — BBBP', fontsize=13, fontweight='bold')
for row, cls in enumerate(['class_0', 'class_1']):
    imgs = sorted(os.listdir(os.path.join(TRAIN_DIR, cls)))[:4]
    for col, fname in enumerate(imgs):
        img = plt.imread(os.path.join(TRAIN_DIR, cls, fname))
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"{cls}\n{LABEL[cls]}", fontsize=8)
        axes[row, col].axis('off')
plt.tight_layout()
plt.savefig(f'samples_bbbp.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Hiperparâmetros Globais

In [ ]:
IMG_HEIGHT  = 224
IMG_WIDTH   = 224
CHANNELS    = 3
BATCH_SIZE  = 32
MAX_EPOCHS  = 50
PATIENCE    = 8
LR_BASE     = 1e-3   # baseline CNN
LR_FT_S1   = 1e-3   # fine-tuning estágio 1 (cabeça congelada)
LR_FT_S2   = 1e-5   # fine-tuning estágio 2 (descongelamento)

print(f"Entrada: {IMG_HEIGHT}x{IMG_WIDTH}x{CHANNELS} | Batch: {BATCH_SIZE} | Épocas máx: {MAX_EPOCHS}")

## 5. Data Generators e Augmentation

**Justificativa:** rotação, flips horizontais/verticais e zoom são transformações
válidas para imagens moleculares (a identidade química não muda).
Mesmo protocolo de augmentation para ambos os modelos — diferença apenas na normalização:
- **Baseline:** `rescale=1/255` (intervalo [0,1])
- **Fine-tuning:** `preprocess_input` do MobileNetV2 (média/desvio ImageNet)


In [ ]:
# ── BASELINE — normalização [0,1] ──────────────────────────────
aug_base = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.1,
    fill_mode='nearest'
)
no_aug_base = ImageDataGenerator(rescale=1./255)

tg_b = aug_base.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_HEIGHT,IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED, shuffle=True)
vg_b = no_aug_base.flow_from_directory(
    VAL_DIR,   target_size=(IMG_HEIGHT,IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED, shuffle=False)
xg_b = no_aug_base.flow_from_directory(
    TEST_DIR,  target_size=(IMG_HEIGHT,IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED, shuffle=False)

# ── FINE-TUNING — preprocess_input MobileNetV2 ─────────────────
aug_ft = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.1,
    fill_mode='nearest'
)
no_aug_ft = ImageDataGenerator(preprocessing_function=preprocess_input)

tg_f = aug_ft.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_HEIGHT,IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED, shuffle=True)
vg_f = no_aug_ft.flow_from_directory(
    VAL_DIR,   target_size=(IMG_HEIGHT,IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED, shuffle=False)
xg_f = no_aug_ft.flow_from_directory(
    TEST_DIR,  target_size=(IMG_HEIGHT,IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED, shuffle=False)

print("Classe 0 =", list(tg_b.class_indices.keys())[0],
      "| Classe 1 =", list(tg_b.class_indices.keys())[1])

## 6. Pesos de Classe (tratamento de desbalanceamento)

In [ ]:
cw_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(tg_b.classes),
    y=tg_b.classes
)
class_weights = {int(i): float(w) for i, w in enumerate(cw_arr)}
print("Pesos de classe:", class_weights)

## 7. Callback Customizado — Tempo por Época

In [ ]:
class TimerCB(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []
        self._t0 = time.time()
    def on_epoch_begin(self, epoch, logs=None):
        self._ep = time.time()
    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.time() - self._ep)
    def on_train_end(self, logs=None):
        self.total   = time.time() - self._t0
        self.mean_ep = float(np.mean(self.times))
        print(f"  Tempo médio/época: {self.mean_ep:.2f}s | Total: {self.total:.2f}s")

## 8. Modelo 1 — CNN Baseline (construída do zero)

**Justificativas de arquitetura:**
- **4 blocos convolucionais** (32→64→128→256 filtros): extração hierárquica de features.
- **BatchNormalization** em cada bloco: estabiliza gradientes e acelera convergência.
- **MaxPooling2D**: redução de dimensionalidade e invariância a translação local.
- **GlobalAveragePooling2D**: mais robusto a variações de posição que Flatten.
- **Dropout(0.5)** + **L2** na cabeça: dupla regularização contra overfitting.
- **EarlyStopping** monitorando `val_auc` (mais informativo que accuracy em datasets desbalanceados).
- **ModelCheckpoint**: preserva os pesos do melhor momento de validação.


In [ ]:
def build_baseline(input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)):
    tf.random.set_seed(SEED)
    m = models.Sequential(name='Baseline_CNN')
    m.add(layers.Input(shape=input_shape))

    for filters in [32, 64, 128]:
        m.add(layers.Conv2D(filters, (3,3), padding='same'))
        m.add(layers.BatchNormalization())
        m.add(layers.Activation('relu'))
        m.add(layers.MaxPooling2D((2,2)))

    # Bloco 4 sem pooling (preserva resolução antes do GAP)
    m.add(layers.Conv2D(256, (3,3), padding='same'))
    m.add(layers.BatchNormalization())
    m.add(layers.Activation('relu'))
    m.add(layers.GlobalAveragePooling2D())

    m.add(layers.Dense(256, activation='relu',
                       kernel_regularizer=regularizers.l2(1e-4)))
    m.add(layers.Dropout(0.5))
    m.add(layers.Dense(1, activation='sigmoid'))

    m.compile(
        optimizer=keras.optimizers.Adam(LR_BASE),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return m

model_b = build_baseline()
model_b.summary()

In [ ]:
CKPT_B = 'best_baseline_bbbp.keras'

cb_b = [
    callbacks.EarlyStopping(monitor='val_auc', patience=PATIENCE,
                            mode='max', restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(CKPT_B, monitor='val_auc', mode='max',
                              save_best_only=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=4,
                                mode='max', min_lr=1e-6, verbose=1),
    TimerCB()
]

print("\n" + "="*55)
print("TREINO — Baseline CNN")
print("="*55)
hist_b = model_b.fit(
    tg_b, epochs=MAX_EPOCHS, validation_data=vg_b,
    class_weight=class_weights, callbacks=cb_b, verbose=1
)
timer_b = cb_b[-1]
n_ep_b  = len(hist_b.history['loss'])
print(f"\nCheckpoint: {CKPT_B} | Épocas: {n_ep_b}")

## 9. Modelo 2 — Fine-Tuning com MobileNetV2 (backbone ImageNet)

**Justificativas:**
- **MobileNetV2** escolhido por: (a) eficiência com depthwise separable convolutions,
  ideal para datasets médios; (b) boa transferência de features (bordas, texturas) para imagens moleculares.
- **Estratégia em 2 estágios:**
  - **Estágio 1** (cabeça congelada, LR=1e-3): inicializa a nova cabeça densa com pesos adequados.
  - **Estágio 2** (últimas 30 camadas descongeladas, LR=1e-5): adapta features de alto nível do backbone
    ao domínio molecular sem destruir features genéricas dos blocos iniciais.


In [ ]:
def build_ft(input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)):
    tf.random.set_seed(SEED)
    base = MobileNetV2(input_shape=input_shape, include_top=False,
                       weights='imagenet', pooling='avg')
    base.trainable = False  # Estágio 1: congelado

    inp = keras.Input(shape=input_shape)
    x   = base(inp, training=False)
    x   = layers.Dense(256, activation='relu',
                        kernel_regularizer=regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m   = keras.Model(inp, out, name='FineTuning_MobileNetV2')
    m.compile(optimizer=keras.optimizers.Adam(LR_FT_S1),
              loss='binary_crossentropy',
              metrics=['accuracy', keras.metrics.AUC(name='auc')])
    return m, base

model_f, base_f = build_ft()
model_f.summary()

In [ ]:
# ── ESTÁGIO 1: Feature Extraction (backbone congelado) ─────────
print("\n" + "="*55)
print("ESTÁGIO 1 — Feature Extraction")
print("="*55)
cb_f1 = [
    callbacks.EarlyStopping(monitor='val_auc', patience=5,
                            mode='max', restore_best_weights=True, verbose=1),
    TimerCB()
]
hist_f1 = model_f.fit(
    tg_f, epochs=15, validation_data=vg_f,
    class_weight=class_weights, callbacks=cb_f1, verbose=1
)
timer_f1 = cb_f1[-1]
print(f"Estágio 1: {len(hist_f1.history['loss'])} épocas")

In [ ]:
# ── ESTÁGIO 2: Fine-Tuning (últimas 30 camadas descongeladas) ──
print("\n" + "="*55)
print("ESTÁGIO 2 — Fine-Tuning (30 camadas descongeladas)")
print("="*55)
base_f.trainable = True
for layer in base_f.layers[:-30]:
    layer.trainable = False
print(f"Parâmetros treináveis: {sum(np.prod(v.shape) for v in model_f.trainable_variables):,}")

model_f.compile(optimizer=keras.optimizers.Adam(LR_FT_S2),
                loss='binary_crossentropy',
                metrics=['accuracy', keras.metrics.AUC(name='auc')])

CKPT_F = 'best_finetune_bbbp.keras'
cb_f2 = [
    callbacks.EarlyStopping(monitor='val_auc', patience=PATIENCE,
                            mode='max', restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(CKPT_F, monitor='val_auc', mode='max',
                              save_best_only=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=4,
                                mode='max', min_lr=1e-7, verbose=1),
    TimerCB()
]
hist_f2 = model_f.fit(
    tg_f, epochs=MAX_EPOCHS, validation_data=vg_f,
    class_weight=class_weights, callbacks=cb_f2, verbose=1
)
timer_f2 = cb_f2[-1]
print(f"\nCheckpoint: {CKPT_F} | Épocas estágio 2: {len(hist_f2.history['loss'])}")

## 10. Curvas de Aprendizado

In [ ]:
def plot_curves(history, title, fname):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for ax, metric, ylabel in zip(
            axes, ['accuracy', 'loss'], ['Acurácia', 'Loss']):
        ax.plot(history.history[metric],         label='Treino',   lw=2)
        ax.plot(history.history[f'val_{metric}'], label='Validação', lw=2, ls='--')
        ax.set_xlabel('Época'); ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel} por Época')
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Salvo: {fname}")

plot_curves(hist_b,
            f'Curvas — Baseline CNN (BBBP)',
            f'curves_baseline_bbbp.png')

In [ ]:
# Mesclar estágios 1 e 2 do fine-tuning para exibição unificada
class FakeHist:
    def __init__(self, h1, h2):
        self.history = {k: h1.history[k] + h2.history.get(k, [])
                         for k in h1.history}

plot_curves(FakeHist(hist_f1, hist_f2),
            f'Curvas — Fine-Tuning MobileNetV2 (BBBP)',
            f'curves_finetune_bbbp.png')

## 11. Avaliação no Conjunto de Teste

In [ ]:
def avaliar(model, xg, tg, vg, nome):
    # Recarregar do checkpoint (garantia)
    tg.reset(); vg.reset(); xg.reset()
    _, acc_tr, _ = model.evaluate(tg, verbose=0)
    _, acc_vl, _ = model.evaluate(vg, verbose=0)
    xg.reset()
    prob = model.predict(xg, verbose=0).ravel()
    y    = xg.classes
    pred = (prob >= 0.5).astype(int)
    acc_te = float(np.mean(pred == y))
    auc_te = float(roc_auc_score(y, prob))
    cm     = confusion_matrix(y, pred)
    print(f"\n{'-'*50}")
    print(f"  {nome}")
    print(f"{'-'*50}")
    print(f"  Acc Treino : {acc_tr:.4f}")
    print(f"  Acc Val    : {acc_vl:.4f}")
    print(f"  Acc Teste  : {acc_te:.4f}")
    print(f"  AUC-ROC    : {auc_te:.4f}")
    print("\nMatriz de Confusão:")
    print(cm)
    print("\nRelatório:")
    print(classification_report(y, pred,
          target_names=['Não penetra a BHE', 'Penetra a BHE']))
    return dict(nome=nome, acc_tr=acc_tr, acc_vl=acc_vl,
                acc_te=acc_te, auc_te=auc_te,
                cm=cm, y=y, prob=prob)

# Carregar melhores checkpoints
print(f"Carregando {CKPT_B} ...")
model_b.load_weights(CKPT_B)
print(f"Carregando {CKPT_F} ...")
model_f.load_weights(CKPT_F)

res_b = avaliar(model_b, xg_b, tg_b, vg_b, 'Baseline CNN')
res_f = avaliar(model_f, xg_f, tg_f, vg_f, 'Fine-Tuning MobileNetV2')

## 12. Matrizes de Confusão e Curva ROC

In [ ]:
# Matrizes de Confusão
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f'Matrizes de Confusão — BBBP', fontsize=13, fontweight='bold')
class_names = ['Não penetra a BHE', 'Penetra a BHE']
for ax, res in zip(axes, [res_b, res_f]):
    tn, fp, fn, tp = res['cm'].ravel()
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"{res['nome']}\nTN={tn} FP={fp} FN={fn} TP={tp}")
    ax.set_xlabel('Predito'); ax.set_ylabel('Real')
plt.tight_layout()
plt.savefig(f'cm_bbbp.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Curva ROC
fig, ax = plt.subplots(figsize=(7, 6))
for res, cor in [(res_b, 'steelblue'), (res_f, 'darkorange')]:
    fpr, tpr, _ = roc_curve(res['y'], res['prob'])
    ax.plot(fpr, tpr, color=cor, lw=2,
            label=f"{res['nome']} (AUC={res['auc_te']:.3f})")
ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
ax.set_title(f'Curva ROC — BBBP', fontsize=13, fontweight='bold')
ax.set_xlabel('Taxa de Falsos Positivos')
ax.set_ylabel('Taxa de Verdadeiros Positivos')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'roc_bbbp.png', dpi=120, bbox_inches='tight')
plt.show()

## 13. Discussão: Falsos Positivos e Falsos Negativos — BBBP

**Falsos Positivos (FP):** compostos classificados como `Penetra a BHE` sendo na verdade `Não penetra a BHE`.
Implica custo experimental desnecessário.

**Falsos Negativos (FN):** compostos classificados como `Não penetra a BHE` sendo na verdade `Penetra a BHE`.
Implica perda de candidatos promissores no pipeline.

O limiar de decisão (0.5 padrão) pode ser ajustado conforme a tolerância ao risco de cada aplicação.


## 14. Tempo de Treinamento — Tabela Comparativa

In [ ]:
n_ep_f1 = len(hist_f1.history['loss'])
n_ep_f2 = len(hist_f2.history['loss'])
n_ep_f  = n_ep_f1 + n_ep_f2
tt_f    = timer_f1.total + timer_f2.total
me_f    = tt_f / n_ep_f

print(f"Métrica                      {' Baseline':>12} {' Fine-Tuning':>14}")
print("-"*56)
print(f"{'Épocas treinadas':<28} {n_ep_b:>12} {n_ep_f:>14}")
print(f"{'Tempo médio/época (s)':<28} {timer_b.mean_ep:>12.2f} {me_f:>14.2f}")
print(f"{'Tempo total (s)':<28} {timer_b.total:>12.2f} {tt_f:>14.2f}")

## 15. Tabela Comparativa Final

In [ ]:
df = pd.DataFrame({
    'Dataset':          ['BBBP','BBBP'],
    'Modelo':           ['Baseline CNN','Fine-Tuning MobileNetV2'],
    'Acc Treino':       [f"{res_b['acc_tr']:.4f}",f"{res_f['acc_tr']:.4f}"],
    'Acc Validação':    [f"{res_b['acc_vl']:.4f}",f"{res_f['acc_vl']:.4f}"],
    'Acc Teste':        [f"{res_b['acc_te']:.4f}",f"{res_f['acc_te']:.4f}"],
    'AUC-ROC Teste':    [f"{res_b['auc_te']:.4f}",f"{res_f['auc_te']:.4f}"],
    'Tempo médio/ép (s)':[f"{timer_b.mean_ep:.2f}",f"{me_f:.2f}"],
    'Tempo total (s)':  [f"{timer_b.total:.2f}",f"{tt_f:.2f}"],
})
print(df.to_string(index=False))
df.to_csv(f'resultados_bbbp.csv', index=False)
print("\nCSV salvo!")

## 16. Análise Comparativa e Conclusões

### Baseline CNN
- Implementação simples e controlada; menor overhead de dependências.
- Maior risco de overfitting em datasets de tamanho médio.
- Ideal quando há grande volume de dados de imagens moleculares.

### Fine-Tuning MobileNetV2
- Aproveita representações pré-aprendidas (bordas, texturas), convergindo mais rápido.
- Tende a generalizar melhor com poucos dados (como neste projeto).
- Exige normalização compatível com ImageNet e maior complexidade de setup.
- O treinamento em 2 estágios protege os pesos do backbone durante a inicialização da cabeça.

### Recomendação para BBBP
Ambas as abordagens fornecem informação complementar.
Fine-tuning é geralmente preferível quando o dataset é pequeno ou médio — situação típica em QSAR.
O modelo baseline serve como referência e pode superar o fine-tuning se o domínio visual das imagens
moleculares diferir muito das imagens naturais do ImageNet.
